In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

In [3]:
train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

100%|██████████| 170M/170M [00:07<00:00, 22.2MB/s]


In [4]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [5]:
class SimpleCIFAR10NN(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),               # (3,32,32) → 3072

            nn.Linear(3072, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 10)          # 10 classes
        )

    def forward(self, x):
        return self.network(x)

In [6]:
model = SimpleCIFAR10NN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [7]:
epochs = 15

train_accuracies = []
test_accuracies = []

for epoch in range(epochs):

    # -----------------
    # Training
    # -----------------
    model.train()
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    train_acc = correct_train / total_train
    train_accuracies.append(train_acc)

    # -----------------
    # Testing
    # -----------------
    model.eval()
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            correct_test += (preds == labels).sum().item()
            total_test += labels.size(0)

    test_acc = correct_test / total_test
    test_accuracies.append(test_acc)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Test Acc: {test_acc:.4f}")

Epoch 1/15 | Train Acc: 0.3799 | Test Acc: 0.4438
Epoch 2/15 | Train Acc: 0.4319 | Test Acc: 0.4680
Epoch 3/15 | Train Acc: 0.4556 | Test Acc: 0.4845
Epoch 4/15 | Train Acc: 0.4689 | Test Acc: 0.4916
Epoch 5/15 | Train Acc: 0.4851 | Test Acc: 0.4886
Epoch 6/15 | Train Acc: 0.4962 | Test Acc: 0.5080
Epoch 7/15 | Train Acc: 0.5049 | Test Acc: 0.5122
Epoch 8/15 | Train Acc: 0.5165 | Test Acc: 0.5056
Epoch 9/15 | Train Acc: 0.5266 | Test Acc: 0.5075
Epoch 10/15 | Train Acc: 0.5320 | Test Acc: 0.5095
Epoch 11/15 | Train Acc: 0.5407 | Test Acc: 0.5211
Epoch 12/15 | Train Acc: 0.5505 | Test Acc: 0.5143
Epoch 13/15 | Train Acc: 0.5557 | Test Acc: 0.5287
Epoch 14/15 | Train Acc: 0.5653 | Test Acc: 0.5302
Epoch 15/15 | Train Acc: 0.5716 | Test Acc: 0.5309
